**This problem set is due Wednesday, November 4, 2026 at 11:59 pm. Please plan ahead and submit your work on time.**

# Problem Set 07: Reinforcement Learning

In this problem set, you will implement model-free approaches for reinforcement learning.


0. [Credit for Contributors (required)](#contributors)

1. [Passive Reinforcement Learning (50 points)](#problem1)
    1. [Direct Evaluation (15 points)](#direct_evaluation)
    2. [Sample Sensitivity (10 points)](#sample_sensitivity)
    3. [Temporal Difference Learning (15 points)](#temporal_difference)
    4. [Learning Rate Sensitivity (10 points)](#learning_rate)
2. [Active Reinforcement Learning (35 points)](#problem2)
    1. [Q-Learning (15 points)](#Qlearning)
    2. [Epsilon-Greedy Q-Learning (10 points)](#epsilon_greedy)
    3. [Exploration vs. Exploitation (10 points)](#exploration)
3. [Exploration Functions vs. Epsilon-Greedy [AI-enabled] (10 points)](#problem3)
    1. [An MDP Where Exploration Functions Win [AI-enabled] (5 points)](#exploration_mdp)
    2. [Visualizing Why Exploration Functions Win [AI-enabled] (5 points)](#exploration_visualization)
4. [Time Spent on Pset (5 points)](#part4)
    
**100 points** total for Problem Set 7

## Imports and Utilities

In [ ]:
# Be sure to run the cell below to import the code needed for this assignment.
from __future__ import division

%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import copy
import networkx as nx
from grid import MDPGrid, generate_mdp_plot, generate_grid_plot
from mdp_utils import *

# imports for autograder
from principles_of_autonomy.grader import Grader
from principles_of_autonomy.notebook_tests.pset_7 import TestPSet7

## <a name="contributors"></a> Credit for Contributors

List the various students, lecture notes, or online resouces that helped you complete this problem set:

Ex: I worked with Bob on the cat reinforcement learning problem.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

## <a name="problem1"></a> 1: Passive Reinforcement Learning (50 points)

In this problem, you will implement the Passive Reinforcement Learning approaches we saw in class: Direct Evaluation and Temporal Difference Learning.

The problems in this problem set are based on the simple MDP class from Problem Set 6 defined in the `mdp_utils.py` file. Feel free to open Problem Set 6 for a reminder of the MDP definition and the grid world our robot operates in. This week, however, we assume that we don't know the transition function T or the reward function R. Instead, we will implement model-free reinforcement learning methods to calculate the Values and/or Q-Values.

We provide you with a `generate_episodes` function that generates episodes from an agent acting according to a policy in the MDP:

```python
def generate_episodes(mdp, policy, num_episodes=5, max_steps=15)
```

Let's build the simple MDP from last week and visualize it again:

In [ ]:
n = 3
goal = (2, 2)
obstacles = [(0, 1)]

# Build MDP with p=0.8 and gamma=0.8. Use default rewards.
mdp = build_mdp(n, 0.8, obstacles, goal, 0.8)

# Visualize the MDP:
# 1. Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()
# 2. Draw goal and obstacle cells.
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k')

Now let's create a policy to follow when we generate episodes:

In [ ]:
# Create simple policy that always moves right.
policy = {s: 'right' for s in mdp.S}

Now let's generate some episodes according to this policy in the MDP:

In [ ]:
episodes = generate_episodes(mdp, policy, num_episodes=10, max_steps=100)

Run the following print block to look at the generated episodes. Notice how they have different lengths because they reach terminal states at different times. Notice also how the stochasticity of the MDP manifests itself in the samples of an episode: sometimes you take an action and end up where you intended; other times the action takes you in a different state. The ratio of these outcomes is implicit from the transition probabilities of the environment.

In [ ]:
# Each episode is a list of (state, action, state', reward) tuples
for i, episode in enumerate(episodes):
    print(f"Episode {i+1} of length {len(episode)}: ".format(i, episode), episode, "\n")

### <a name="direct_evaluation"></a> 1A. Direct Evaluation (15 points)

In this part of the problem, you will implement **Direct Evaluation** (also called Monte-Carlo Evaluation). Your function should have the following signature:
```python
def direct_evaluation(episodes: list, gamma: float) -> dict
```

The function takes in a list of episodes representing different experiences of the agent acting according to a fixed policy in the MDP, and the discount factor gamma. The function should return a Python dictionary with the estimated value for each state in the MDP.

In general Direct Evaluation has 2 variants: First-Visit and Every-Visit. In First-Visit, we only add a sample of discounted rewards to the value estimate whenever we encounter a state for the first time in the episode. In Every-Visit, we add a sample *every* time we encounter a state (even if it's repeated). You should implement the First-Visit variant of Direct Evaluation that we talked about in class. For this, you'll find it useful to keep track of states visited so far in the episode with a `visited_states` list (that you reset whenever you move on to the next episode).

<div class="alert alert-info">
Implement the function `direct_evaluation(episodes, gamma)` below.
</div>

In [ ]:
def direct_evaluation(episodes: list, gamma: float) -> dict:
    """
    Direct Evaluation method to estimate the value function for each state for a fixed policy.
    
    Args:
        episodes (list): A list of episodes. Each episode is a list of (state, action, state', reward) tuples.
        gamma (float): The discount factor.
        
    Returns:
        V: A dictionary mapping states to their estimated value V(s).
    """
    raise NotImplementedError()

Check out how your code performs on the initial grid world we defined:

In [ ]:
# Test your code for direct evaluation with the example from before and a policy that always goes right.

# Build MDP with p=0.8 and gamma=0.8. Use default rewards.
n = 3
goal = (2, 2)
obstacles = [(0, 1)]
mdp = build_mdp(n, 0.8, obstacles, goal, 0.8)

# Create simple policy that always moves right.
policy = {s: 'right' for s in mdp.S}

# Generate some episodes.
episodes = generate_episodes(mdp, policy, num_episodes=100000, max_steps=100)

# Perform direct evaluation with these episodes.
V = direct_evaluation(episodes, mdp.gamma)

# Visualize values:
# 1. Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()
# 2. Plot values with colors and numbers.
g.plot_V(axes, V, print_numbers=True)
# 3. Draw goal and obstacle cells
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k')

You can convince yourself this is correct by running Policy Evaluation in the PSet 6 notebook with this always-go-right policy and comparing the values.

In [ ]:
"""Test your direct evaluation code here."""
Grader.run_single_test_inline(TestPSet7, "test_1_direct_evaluation", locals())

### <a name="sample_sensitivity"></a> 1B. Sample Sensitivity (10 points)

In class, we discussed that Direct Evaluation, while simple, has a number of limitations. We now want to observe its performance with varying sample sizes. Run the MDP example above with a varying number of samples; concretely, run the block with different values for `num_episodes` inside `generate_episodes`. Try 100, 1000, 10000, and 100000. For each `num_episodes` value run the block a few times, then answer the following:

1. What trends do you observe in the value estimates as you increase the number of episodes?
2. How does the number of samples affect the convergence of the value function?
3. Why does Direct Evaluation require a large number of samples to provide accurate estimates?


<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

raise NotImplementedError()

### <a name="temporal_difference"></a> 1C. Temporal Difference Learning (15 points)

In this part of the problem, you will implement **Temporal Difference Learning**, which loops over the episode samples and performs the TD update to the value function one sample at a time. Your function should have the following signature:
```python
def td_learning(episodes: list, gamma: float, alpha: float) -> dict
```

The function takes in a list of episodes experienced by following the fixed policy in the MDP, the discount factor gamma, and the learning rate alpha for blending past values with the new sample. The function should return a Python dictionary with the estimated value for each state in the MDP.

<div class="alert alert-info">
Implement the function `td_learning(episodes, gamma, alpha)` below.
</div>

In [ ]:
def td_learning(episodes: list, gamma: float, alpha: float) -> dict:
    """
    Temporal-Difference Learning to estimate the value function for a fixed policy.
    
    Args:
        episodes (list): A list of episodes. Each episode is a list of (state, action, state', reward) tuples.
        gamma (float): The discount factor of the MDP.
        alpha (float): The learning rate.
        
    Returns:
        V: A dictionary mapping states to their estimated value V(s).
    """
    raise NotImplementedError()

Check out how your code performs on the initial grid world we defined:

In [ ]:
# Test your code for TD Learning with the example from before and a policy that always goes right.

# Build MDP with p=0.8 and gamma=0.8. Use default rewards.
n = 3
goal = (2, 2)
obstacles = [(0, 1)]
mdp = build_mdp(n, 0.8, obstacles, goal, 0.8)

# Create simple policy that always moves right.
policy = {s: 'right' for s in mdp.S}

# Generate some episodes.
episodes = generate_episodes(mdp, policy, num_episodes=100000, max_steps=100)

In [ ]:
# Perform TD learning with these episodes.
alpha = 0.01
V = td_learning(episodes, mdp.gamma, alpha)

# Visualize values:
# 1. Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()
# 2. Plot values with colors and numbers.
g.plot_V(axes, V, print_numbers=True)
# 3. Draw goal and obstacle cells
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k')

In [ ]:
"""Test your TD learning code here."""
Grader.run_single_test_inline(TestPSet7, "test_2_TD_learning", locals())

### <a name="learning_rate"></a> 1D. Learning Rate Sensitivity (10 points)

In the code above, we ran TD Learning with alpha = 0.01. Try experimenting with more values of alpha: 0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5. Answer the following based on what you observe:

1. How does a very low learning rate (e.g., alpha = 0.00001) affect the value function compared to a moderate learning rate (e.g., alpha = 0.01)? What about a very high learning rate (e.g., alpha = 0.5)?
2. Identify a learning rate that seems to work well for your specific MDP setup and explain why you think it strikes the right balance.
3. Reflecting on your results, how do you think the choice of learning rate might affect the performance of TD Learning in more complex environments?

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

raise NotImplementedError()

## <a name="problem2"></a> 2: Active Reinforcement Learning (35 points)

Passive Reinforcement Learning is useful for evaluating fixed policies without explicit knowledge of T and R. However, it can't be used for turning these values into a new (improved) policy. For that, we need active interactions with the environment (not just passive reflection on past episodes). In this problem, we switch over to Active Reinforcement Learning, specifically to Q-Learning, and we will investigate the exploration-exploitation tradeoff and how that can affect the algorithm's efficiency.

Since we are interacting with the world in real time now, instead of using `generate_episodes` to generate experiences to learn from, we will use a function `sample_environment` which given a state and an action it queries the MDP environment for the resulting next state and reward:

```python
def sample_environment(mdp: MDP, state: tuple, action: str) -> tuple[tuple, float]
```

You can try running the following block to see how to call `sample_environment` and what it returns. Try running it a few times (and possibly with different states and actions) to convince yourself that there is stochasticity involved. (Note if you runt the following cell after running code for Q-learning below you may find that it's no longer stochastic because we set a random seed in that part for grading.)

In [ ]:
# Build MDP with p=0.8 and gamma=0.8. Use default rewards.
n = 3
goal = (2, 2)
obstacles = [(0, 1)]
mdp = build_mdp(n, 0.8, obstacles, goal, 0.8)

state = (1, 2)
action = "right"
next_state, reward = sample_environment(mdp, state, action
                                       )
print(state, action, next_state, reward)

Before we dive into implementing Q-Learning, let's first see some Q-values. We will store Q-values as a Python dictionary that maps each state in the MDP to another dictionary that maps each action to the corresponding Q-value. To see an example, here's what a random Q-Value function looks like:

In [ ]:
# A random Q-value function:
def random_Q(states: list[tuple], actions: list[str]):
    # Initialize empty Q-values dictionary.
    Q = {}
    # Generate random Q-values between minV and maxV.
    minQ, maxQ = -100, 100
    for state in states:
        Q[state] = {}
        for action in actions:
            Q[state][action] = np.random.uniform(minQ, maxQ)
    return Q

# Print values generated with the random Q-Value function.
print("Q_random is a valid Q-Value function, although definitely not optimal (it's just random!)")
n = 3
mdp = build_mdp(n, 0.8, [], (1, 1), 0.8)
Q_random = random_Q(mdp.S, mdp.A)
Q_random

Let's now visualize these Q-values in the grid:

In [ ]:
# Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()

# Plot Q in the grid.
g.plot_Q(axes, Q_random, print_numbers = True)

### <a name="Qlearning"></a> 2A. Q-Learning (15 points)

You will now implement **Q-Learning**. While Direct Evaluation and TD Learning were on-policy (they relied on samples coming from a fixed policy), Q-Learning is off-policy (doesn't depend on samples from one fixed policy), can choose its own actions as we go, and can converge to the optimal policy even if the agent is acting suboptimally. You should thus make use of `sample_environment` in your code to try out actions in the environment. For this problem, you should use a random action selection strategy meaning you should choose randomly from the available actions to sample the environment.

Your function should have the following signature:
```python
def q_learning(num_episodes: int, mdp: MDP, gamma: float, alpha: float, max_steps: int = 100)
```

The function takes in the number of episodes your agent can attempt in the MDP, the discount factor gamma, the learning rate alpha (similar to TD learning), and a maximum length for each episode (to avoid infinite episodes). The function should return a Python dictionary with the estimated Q-value for each (state, action) pair in the MDP, as well as a policy that results from maximizing the final values at each state.

To help you out, we provide some starter code that initializes an empty Q-value dictionary, loops through episodes, and starts an episode at a random state. The function `mdp.is_sink_state(state)` checks whether a state is terminal or not; the episode ends right after we enter a terminal state or when we reach the maximum allowable length of the episode max_steps.

<div class="alert alert-info">
Implement the function `q_learning(num_episodes, gamma, alpha, max_steps=100)` below.

Complete the two To dos.
</div>

In [ ]:
def q_learning(num_episodes: int, mdp: MDP, gamma: float, alpha: float, max_steps: int = 100):
    """
    Active Q-Learning with random action selection.
    
    Args:
        num_episodes (int): The number of episodes to run.
        gamma (float): The discount factor of the MDP.
        alpha (float): The learning rate.
        max_steps (int): The maximum number of steps per episode to avoid infinite loops.
        
    Returns:
        Q: A dictionary mapping states to another dictionary mapping actions to their estimated Q-values.
        policy: A dictionary mapping states to actions to take.
    """
    # in real life, you won't have access to mdp. We are including it as a parameter for testing purposes.
    # DO NOT USE mdp.R or mdp.T!

    # Initialize Q-value dictionary.
    Q = {state: {action: 0.0 for action in mdp.A} for state in mdp.S}
    
    # Loop through episodes.
    for episode in range(num_episodes):
        # Start episode in a random state.
        state = random.choice(list(mdp.S))
        # Track number of steps in episode.
        steps = 0
        # While the episode is not done.
        while not mdp.is_sink_state(state) and steps < max_steps:
            steps += 1

            # Pick an action randomly
            action = random.choice(mdp.A)

            ### TO DO 1: sample the environment, update Q-values using the Q-learning update rule, and transition to the next state.
            raise NotImplementedError()

    ### TO DO 2: Derive policy from Q-values by maximizing over actions at each state.
    raise NotImplementedError()
    
    return Q, policy

Check out how your code performs on the initial grid world we defined:

In [ ]:
# Test your code for Q Learning with the example from before.

# Build MDP with p=0.8 and gamma=0.8. Use default rewards.
n = 3
goal = (2, 2)
obstacles = [(0, 1)]
mdp = build_mdp(n, 0.8, obstacles, goal, 0.8)

# Perform Q learning with these episodes.
alpha = 0.01
num_episodes = 10000
Q, policy = q_learning(num_episodes, mdp, mdp.gamma, alpha)

# Visualize values:
# 1. Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()
# 2. Plot values with colors and numbers.
g.plot_Q(axes, Q, print_numbers=True)
# 3. Draw goal and obstacle cells
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k')   

Convince yourself that the optimal policy is the same one we got with Policy Extraction when we knew the ground truth T and R values in Problem Set 4.

In [ ]:
# Visualize the result.
# 1. Create grid for plotting
g = MDPGrid(n, n)
axes = g.draw()
# 2. Plot the values and the policy.
g.plot_policy(axes, policy)
# 3. Draw goal and obstacle cells
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k') 

In [ ]:
"""Test your Q learning code here."""
Grader.run_single_test_inline(TestPSet7, "test_3_Q_learning", locals())

### <a name="epsilon_greedy"></a> 2B. Epsilon-Greedy Q-Learning (10 points)

We saw in class that an agent that selects actions randomly will take longer to explore the space and converge than one that has a more clever exploration strategy. We will now implement an **Epsilon-Greedy** variant of Q-Learning, meaning it chooses random actions an epsilon fraction of the time, and follows its current best Q-values otherwise. Note that choosing a random action may result in choosing the best action - that is, you should not choose a random sub-optimal action, but rather any random legal action.

Your function should have the following signature:
```python
def q_learning_epsilon_greedy(num_episodes: int, mdp: MDP, gamma: float, alpha: float, epsilon: float, max_steps: int = 100) -> dict
```

The function takes in the number of episodes your agent can attempt in the MDP, the discount factor gamma, the learning rate alpha, the epsilon exploration probability, and the maximum steps of an episode. The function should return a Python dictionary with the estimated Q-value for each state-action pair in the MDP, as well as a policy that results from maximizing the final values at each state.

A good place to start is to just copy your code for `q_learning` below. 

**Important Notes**

It's important to do the following for passing the test.
- Please use `action = random.choice(mdp.A)` to generate the action for each step as we provided above in q-learning. 
- For the **Epsilon-Greedy**  part, please use `if random.uniform(0, 1) < epsilon:` then choose random action otherwise act on current policy.
- You only need < 5 lines of change to your code.

<div class="alert alert-info">
Implement the function `q_learning_epsilon_greedy(num_episodes, gamma, alpha, epsilon, max_steps=100)` below.
</div>

In [ ]:
def q_learning_epsilon_greedy(num_episodes: int, mdp: MDP, gamma: float, alpha: float, epsilon: float, max_steps: int = 100) -> dict:
    """
    Active Q-Learning with epsilon-greedy action selection.
    
    Args:
        num_episodes (int): The number of episodes to run.
        gamma (float): The discount factor.
        alpha (float): The learning rate.
        epsilon (float): Exploration probability.
        max_steps (int): The maximum number of steps per episode.
        
    Returns:
        Q: A dictionary mapping states to another dictionary mapping actions to their estimated Q-values.
        policy: A dictionary mapping states to the optimal actions.
    """
    # in real life, you won't have access to mdp. We are including it as a parameter for testing purposes.
    # BEGIN HERE!
    raise NotImplementedError()


Check out how your code performs on the initial grid world we defined:

In [ ]:
# Test your code for Epsilon Greedy Q Learning with the example from before.

# Build MDP with p=0.8 and gamma=0.8. Use default rewards.
n = 3
goal = (2, 2)
obstacles = [(0, 1)]
mdp = build_mdp(n, 0.8, obstacles, goal, 0.8)

# Perform Q learning with these episodes.
alpha = 0.01
epsilon = 0.5
num_episodes = 10000
Q, policy = q_learning_epsilon_greedy(num_episodes, mdp, mdp.gamma, alpha, epsilon)

# Visualize values:
# 1. Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()
# 2. Plot values with colors and numbers.

g.plot_Q(axes, Q, print_numbers=True)
# 3. Draw goal and obstacle cells
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k')   

In [ ]:
"""Test your Epsilon Greedy Q learning code here."""
Grader.run_single_test_inline(TestPSet7, "test_4_Q_learning", locals())

### <a name="exploration"></a> 2C. Exploration vs. Exploitation (10 points)

We will now run your code on the larger grid world with more obstacles. Execute the following code:


In [ ]:
n = 10
goal = (5,8)
obstacles = [(1,3), (9,0), (8,8)] + \
            [(4, 2), (4, 3), (4, 6)] + \
            [(6, 2), (6, 3), (6, 5), (6, 6)]
mdp = build_mdp(n, p=0.8, obstacles=obstacles, goal=goal, gamma=0.8, goal_reward=100, obstacle_reward=-500)

# Perform Q learning with these episodes.
alpha = 0.01
epsilon = 0.1
num_episodes = 50000
Q, policy = q_learning_epsilon_greedy(num_episodes, mdp, mdp.gamma, alpha, epsilon)

# Visualize values:
# 1. Create grid for plotting.
g = MDPGrid(n, n)
axes = g.draw()
# 2. Plot values with colors and numbers.
g.plot_Q(axes, Q, print_numbers=False)
g.plot_policy(axes, policy)
# 3. Draw goal and obstacle cells
g.draw_cell_circle(axes, goal, color='g')
for ob in obstacles:
    g.draw_cell_circle(axes, ob, color='k')   

Now try running the same code block with different values of epsilon: 0 (pure exploitation), 0.1, 0.5, 1.0 (pure exploration). Try also varying num_episodes to get an idea of how many samples the algorithm needs to converge with different epsilon values. Answer the following questions:

1. How does changing the epsilon parameter affect the performance and policy learned by the agent?
2. Can you find a balance (pick a value for epsilon) between exploration and exploitation that leads to both fast convergence and a good policy?

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

raise NotImplementedError()

## <a name="problem3"></a> 3. Exploration Functions vs. Epsilon-Greedy [AI-enabled] (10 points)

In Problem 2C, epsilon-greedy Q-learning explored by acting randomly a fixed fraction of the time. In lecture, we discussed an alternate approach, _exploration functions_, which encourage the agent to explore the environment in a more systematic way. This problem *explores* (get it?) the difference between these two approaches.

### <a name="exploration_mdp"></a> 3A. Finding an MDP Where Exploration Functions Win [AI-enabled] (5 points)


Note: like in Problem Set 6, we will use MDPs on graphs, built with `build_graph_mdp(G, goal, p, gamma, goal_reward=100)`. 
<details>
<summary>Click here for a reminder (condensed from Problem Set 6).</summary>
    Each edge of the graph is an action. If the agent plays an action whose edge touches its current state, it moves along that edge to the other endpoint with probability <i>p</i>, and stays in its current state with probability 1-<i>p</i>. Should the agent play an action that corresponds to an edge that is not associated with the current state, then it deterministically stays in the same state. The agent receives a reward of $+100$ when it arrives at the goal state, which is terminal, and *0* for every other transition.
</details>


Recall the exploration function from lecture, which takes a Q-value estimate $u$ and a visit count $n$, and returns an optimistic utility:

$$f(u, n) = u + \frac{k}{n + 1}.$$

(We use $n + 1$ instead of $n$ so that unvisited state-action pairs have a finite bonus of $k$.) Q-learning with an exploration function keeps a count $N(s, a)$ of how many times it has taken action $a$ in state $s$, always acts greedily with respect to $f(Q(s, a), N(s, a))$, and uses the modified Q-update

$$Q(s, a) \leftarrow (1 - \alpha)\, Q(s, a) + \alpha \left[ R(s, a, s') + \gamma \max_{a'} f(Q(s', a'), N(s', a')) \right].$$


Using an LLM (ChatGPT, Claude, etc.) and the `networkx` library, define an MDP and a start state so that Q-learning with the exploration function learns a near-optimal policy for at least 8 of 10 seeds, while epsilon-greedy Q-learning does so for at most 2 of 10 seeds. Some more constraints on the MDP:

- the MDP is built with `build_graph_mdp`, using $p = 0.8$ and $\gamma = 0.95$, on a _tree_ (a connected graph with $|A| = |S| - 1$ edges, like in Problem Set 6) with at most 16 nodes,
- the start state is not the goal state,
- when both agents are trained with $\alpha = 0.1$, $\epsilon = 0.1$, $k = 1$, 500 episodes, and at most 50 steps per episode

Lastly, we say a learned policy $\pi$ is _near-optimal_ if, when evaluated exactly on the true MDP, $V^\pi(\text{start}) \geq 0.9\, V^*(\text{start})$.



The autograder runs its own implementation of epsilon-greedy Q-learning (as in Problem 2B) and Q-learning with the exploration function above. Each implementation differs from Problem 2 in two ways: every episode starts from the same `start` state, instead of a random state, and ties between equally good actions are broken uniformly at random, instead of by the order of `mdp.A`. Each instance of Q-learning is executed with 10 random seeds. 
Running the test cell below prints, for each agent, how many seeds learned a near-optimal policy (defined below), so you can check your MDP before submitting. It also saves a record of every training run to `exploration_traces.pkl`, which you will use for a visualization in 3B.



<div class="alert alert-info">
Implement the function `build_exploration_mdp()`, which returns a tuple `(mdp, start)`, below. Include any LLM-generated code you used to find your MDP.
</div>

In [ ]:
def build_exploration_mdp() -> tuple[MDP, object]:
    """
    Return (mdp, start), where mdp is built with build_graph_mdp(G, goal, p=0.8, gamma=0.95)
    on a tree G with at most 16 nodes, and start is a state of mdp other than the
    goal, such that Q-learning with an exploration function learns a near-optimal policy from
    start for at least 8 of 10 seeds, while epsilon-greedy Q-learning does so for at most 2.
    """
    raise NotImplementedError()

In [ ]:
mdp, start = build_exploration_mdp()
print(f"{len(mdp.S)} states, {len(mdp.A)} actions, start = {start}")

In [ ]:
Grader.run_single_test_inline(TestPSet7, "test_5_exploration_mdp", locals())

### <a name="exploration_visualization"></a> 3B. Visualize an Explanation for Why Exploration Functions Win [AI-enabled] (5 points)

Using an LLM (ChatGPT, Claude, etc.) and the `matplotlib` library, create a visualization that argues why Q-learning with an exploration function succeeds on your MDP from 3A while epsilon-greedy Q-learning fails. The visualization must articulate an entire argument as to how the structure of your MDP, and the way each agent chooses its actions, leads to the difference.

You do not need to train the agents yourself: running the test in 3A saves a record of every training run to `exploration_traces.pkl`. Load it with:

```python
import pickle
with open("exploration_traces.pkl", "rb") as f:
    traces = pickle.load(f)
```

`traces` is a dictionary with the following keys:
- `"start"`, `"goal"`, and `"V_star_start"`: the start state, the goal state, and $V^*(\text{start})$ for your MDP,
- `"params"`: the hyperparameters used for training ($\alpha$, $\epsilon$, $k$, the number of episodes, and the maximum number of steps per episode),
- `"epsilon_greedy"` and `"exploration_function"`: for each agent, a list with one entry per seed. Each entry is a dictionary with:
    - `"episodes"`: every training episode, as a list of `(state, action, state', reward)` tuples (the same format as `generate_episodes` in Problem 1),
    - `"policy"`: the greedy policy learned at the end of training,
    - `"value_history"`: a list of `(episode, value)` pairs, giving $V^\pi(\text{start})$ of the greedy policy every 10 episodes of training,
    - `"near_optimal"`: whether the final policy is near-optimal.

Some ideas:
- Like the "2D point" example from lecture, show where each agent spends its time: e.g., color the nodes of your graph by how often each agent visited them in `"episodes"`.
- Plot `"value_history"` for each agent, compared with $V^*(\text{start})$.
- Show the episode at which each agent first reaches the goal, across seeds.

<div class="alert alert-info">
Implement the code that plots and displays your visualization in the cell below.
</div>

In [ ]:
# visualization code here.

## BEGIN DELETE
# Visualize why the exploration function helps on the 15-state path.
# Run the 3A test cell first; it creates exploration_traces.pkl.
import pickle
from collections import Counter
from matplotlib.lines import Line2D

with open("exploration_traces.pkl", "rb") as f:
    traces = pickle.load(f)

start = traces["start"]
goal = traces["goal"]
G = nx.path_graph(15)  # Same graph returned by build_exploration_mdp().
agents = ["epsilon_greedy", "exploration_function"]
agent_labels = {
    "epsilon_greedy": r"$\epsilon$-greedy",
    "exploration_function": "Exploration function",
}
agent_colors = {
    "epsilon_greedy": "tab:orange",
    "exploration_function": "tab:blue",
}
runs = {agent: traces[agent] for agent in agents}
num_near_optimal = {
    agent: sum(bool(run.get("near_optimal", False)) for run in runs[agent])
    for agent in agents
}

# Count actual state visits across all episodes and seeds. Each trajectory's
# initial state is counted once, followed by each recorded next state.
def pooled_state_visits(agent_runs):
    visits = Counter({state: 0 for state in G.nodes})
    for run in agent_runs:
        for episode in run.get("episodes", []):
            if not episode:
                continue
            visits[episode[0][0]] += 1
            for transition in episode:
                visits[transition[2]] += 1
    return visits

visits = {agent: pooled_state_visits(runs[agent]) for agent in agents}
max_visits = max(1, max(max(v.values(), default=0) for v in visits.values()))

# Infer the horizon from saved records, falling back to the assignment's 500.
episode_limit = max(
    (len(run.get("episodes", [])) for agent in agents for run in runs[agent]),
    default=500,
)
episode_limit = max(1, episode_limit)

fig, axs = plt.subplots(2, 2, figsize=(17, 10))
fig.suptitle("Why systematic exploration wins on a long, sparse-reward chain", fontsize=16)

# (a,b) The same path, colored by pooled state visits for each algorithm.
pos = {state: (state, 0.0) for state in G.nodes}
for ax, agent in zip(axs[0], agents):
    node_counts = [visits[agent][state] for state in G.nodes]
    nx.draw_networkx_edges(G, pos, ax=ax, width=1.5, edge_color="0.45")
    nx.draw_networkx_nodes(
        G, pos, ax=ax, node_color=node_counts, cmap=plt.cm.viridis,
        vmin=0, vmax=max_visits, node_size=360, edgecolors="black", linewidths=0.8,
    )
    labels = {state: str(state) for state in G.nodes}
    labels[start] = f"S\n{start}"
    labels[goal] = f"G\n{goal}"
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=8)
    # Put pooled visit counts under the nodes, so the colors are interpretable.
    def compact_count(value):
        return f"{value / 1000:.1f}k" if value >= 1000 else str(value)
    for state in G.nodes:
        ax.text(state, -0.34, compact_count(visits[agent][state]), ha="center", va="top", fontsize=7)
    # Label the globally available edge-actions.
    for edge_index, (u, v) in enumerate(G.edges()):
        ax.text((u + v) / 2, 0.22, f"e{edge_index}", ha="center", va="bottom", fontsize=7, color="0.3")
    ax.set_title(
        f"{agent_labels[agent]}: {num_near_optimal[agent]}/{len(runs[agent])} near-optimal seeds"
    )
    ax.set_xlim(-0.65, len(G.nodes) - 0.35)
    ax.set_ylim(-0.62, 0.58)
    ax.axis("off")

# (c) Greedy policy value at the start throughout training.
ax = axs[1, 0]
x_grid = np.arange(0, episode_limit + 1, 10, dtype=float)
for agent in agents:
    histories = []
    for run in runs[agent]:
        history = np.asarray(run.get("value_history", []), dtype=float)
        if history.ndim != 2 or history.shape[0] == 0 or history.shape[1] < 2:
            continue
        history = history[np.argsort(history[:, 0])]
        # Interpolate to a shared episode grid for a per-seed comparison.
        histories.append(np.interp(x_grid, history[:, 0], history[:, 1]))
        ax.plot(history[:, 0], history[:, 1], color=agent_colors[agent], alpha=0.16, linewidth=0.9)
    if histories:
        values = np.vstack(histories)
        median = np.median(values, axis=0)
        low, high = np.percentile(values, [10, 90], axis=0)
        ax.plot(x_grid, median, color=agent_colors[agent], linewidth=2.4, label=agent_labels[agent])
        ax.fill_between(x_grid, low, high, color=agent_colors[agent], alpha=0.18, linewidth=0)

v_star = float(traces["V_star_start"])
ax.axhline(v_star, color="black", linestyle="--", linewidth=1.5, label=r"$V^*(start)$")
ax.axhline(0.9 * v_star, color="black", linestyle=":", linewidth=1.3, label=r"$0.9V^*(start)$ threshold")
ax.set_title("Greedy policy value at the start")
ax.set_xlabel("Training episode")
ax.set_ylabel(r"$V^\pi(start)$")
ax.set_xlim(0, episode_limit)
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="best")

# (d) First episode in which each training run reaches the goal.
ax = axs[1, 1]
first_hits = {}
for agent in agents:
    hits = []
    for run in runs[agent]:
        first_hit = None
        for episode_index, episode in enumerate(run.get("episodes", []), start=1):
            if any((transition[2] == goal) or (transition[3] > 0) for transition in episode):
                first_hit = episode_index
                break
        hits.append(first_hit)
    first_hits[agent] = hits

not_reached_y = episode_limit + max(15, 0.04 * episode_limit)
for offset, agent in zip((-0.12, 0.12), agents):
    for seed_index, hit in enumerate(first_hits[agent], start=1):
        x = seed_index + offset
        if hit is None:
            ax.scatter(x, not_reached_y, marker="x", s=55, color=agent_colors[agent], linewidths=2)
        else:
            ax.scatter(x, hit, marker="o", s=42, color=agent_colors[agent], edgecolors="black", linewidths=0.4)
    ax.scatter([], [], marker="o", s=42, color=agent_colors[agent], label=agent_labels[agent])
ax.axhline(episode_limit, color="0.35", linestyle="--", linewidth=1)
ax.set_title("Episode of first goal visit, by seed")
ax.set_xlabel("Random seed")
ax.set_ylabel("First goal episode")
ax.set_xticks(range(1, max(len(runs[a]) for a in agents) + 1))
ax.set_ylim(0, not_reached_y + max(8, 0.025 * episode_limit))
ax.grid(alpha=0.25)
legend_handles = [
    Line2D([], [], marker="o", linestyle="None", markersize=6,
           markerfacecolor=agent_colors[a], markeredgecolor="black", label=agent_labels[a])
    for a in agents
]
legend_handles.append(
    Line2D([], [], marker="x", linestyle="None", markersize=7,
           color="0.25", label="Goal not reached within training")
)
ax.legend(handles=legend_handles, fontsize=8, loc="upper left")

fig.text(
    0.5, 0.015,
    "Mechanism: all 14 edge-actions are available at every state, but only incident edges move the agent; other choices self-loop. "
    "When Q-values are similar, f(Q,N)=Q+1/(N+1) favors untried state-action pairs, so the exploration agent systematically tests alternatives. "
    "Epsilon-greedy samples with replacement and does not prioritize untried pairs; on this long chain, many 50-step episodes can end before the distant reward is found.",
    ha="center", va="bottom", fontsize=9, wrap=True,
)
fig.subplots_adjust(left=0.04, right=0.985, bottom=0.14, top=0.90, wspace=0.22, hspace=0.42)
plt.show()

print(
    "Near-optimal policies: "
    + ", ".join(
        f"{agent_labels[a]} {num_near_optimal[a]}/{len(runs[a])}" for a in agents
    )
)
print("First goal episode by seed:")
for agent in agents:
    print(f"  {agent_labels[agent]}: {first_hits[agent]}")
## END DELETE

## <a name="part4"></a> 4. Time Spent on Pset (5 points)

Please use [this form](https://forms.gle/pbng9KntwM5V6Fr46) to tell us how long you spent on this pset. After you submit the form, the form will give you a confirmation word. Please enter that confirmation word below to get an extra 5 points. 

In [ ]:
form_confirmation_word = "ENTER_THE_CONFIRMATION_WORD_HERE"

In [ ]:
# Run all tests
Grader.grade_output([TestPSet7], [locals()], "results.json")
Grader.print_test_results("results.json")

In [ ]:
###############################################
#   SAVE YOUR NOTEBOOK BEFORE SUBMITTING!!!   #
###############################################

In [ ]:
# Make sure you save the notebook before running this cell so that the most updated version is zipped!
Grader.prepare_submission("ProblemSet07_RL_release")